In [9]:
from __future__ import annotations

import pathlib
from collections import defaultdict

import numpy as np
import pandas as pd
from kebab.utils.dataset.wikidata.wikidata_utils import ResolvedWikidataEntity

In [2]:
dataset_path = (
    pathlib.Path.home()
    / "OneDrive - Microsoft"
    / "Benchmark"
    / "Datasets"
    / "REBEL"
    / "Fragments Resolved"
    # / "sample"
    / "rebel_entity_fragments.jsonl"
)

Load the fragments

In [3]:
fragments = []

# we will not be using fragments with no names
fragments_with_no_names = 0

# load the data
with open(dataset_path, encoding="utf-8") as f:
    for line in f:
        fragment = ResolvedWikidataEntity.from_json(line.strip())

        if not fragment.names:
            fragments_with_no_names += 1
            del fragment
            continue

        # reduce memory footprint
        if "doc_id" in fragment.metadata:
            del fragment.metadata["doc_id"]

        if "source_text_hash" in fragment.metadata:
            del fragment.metadata["source_text_hash"]

        if "fragment_id" in fragment.metadata:
            del fragment.metadata["fragment_id"]

        fragment.evidence_map = None
        fragment.source_ids = None

        fragments.append(fragment)

print(f"Loaded {len(fragments):,d} fragments, ignored {fragments_with_no_names:,d} fragments with no names")

Loaded 6,559,239 fragments, ignored 0 fragments with no names


Example fragment

In [4]:
fragments[0]

ResolvedWikidataEntity(entity_id='Q33298', properties=defaultdict(<class 'list'>, {'name': ['Filipino']}), source_ids=None, evidence_map=None, metadata={'type': ['national language', 'modern language', 'standard variety', 'dialect', 'dialect group', 'logographic writing system', 'multiethnolect']})

In [5]:
# compute counts of property occurrence and entity types of the fragments
property_counts = defaultdict(int)
type_counts = defaultdict(int)

for fragment in fragments:
    for prop_name, prop_value in fragment.properties.items():
        if prop_value:
            property_counts[prop_name] += 1

    for ent_type in fragment.wikidata_type:
        type_counts[ent_type] += 1

Top properties by occurrence in the fragments
---

In [6]:
df = (
    pd.DataFrame(property_counts.items(), columns=["property", "count"])
    .sort_values(by="count", ascending=False)
    .reset_index(drop=True)
)
df.to_csv("property_occurrence.csv", index=False)
df[:20]

,property,count
0,name,6559239
1,located in the administrative territorial entity,604570
2,date of birth,588646
3,country,432616
4,instance of,278493
5,date of death,239364
6,sport,234898
7,publication date,154761
8,point in time,152938
9,place of birth,142728


Top entity types of the fragments
---

In [7]:
df = (
    pd.DataFrame(type_counts.items(), columns=["type", "count"])
    .sort_values(by="count", ascending=False)
    .reset_index(drop=True)
)
df.to_csv("type_occurrence.csv", index=False)
df[:20]

,type,count
0,human,1558003
1,taxon,261278
2,film,137641
3,album,111664
4,corporation,100613
5,sports series,96986
6,village,80457
7,human settlement,77198
8,musical group,61526
9,literary work,53860


Properties overlap
---

In [11]:
# for each property how often that two distinct fragments have (1) a value for this property, and (2) the same value this property
property_value_counts = defaultdict(lambda: defaultdict(int))
entity_value_counts = defaultdict(int)

for entity in fragments:
    for prop_name, values in entity.properties.items():
        entity_value_counts[prop_name] += 1
        for value in values:
            property_value_counts[prop_name][value] += 1

entity_count = len(fragments)

rows = []
for prop_name, value_counts in property_value_counts.items():
    arr = np.array(list(value_counts.values()))
    ent_val_count = entity_value_counts[prop_name]
    ent_probs = arr / entity_count
    cond_ent_probs = arr / ent_val_count
    prob = (ent_probs**2).sum()
    cond_prob = (cond_ent_probs**2).sum()
    ent_fraction = ent_val_count / entity_count
    rows.append((prop_name, len(value_counts), prob, cond_prob, ent_fraction))

overlap_df = pd.DataFrame(
    rows, columns=["property", "distinct_value_count", "overlap_prob", "cond_overlap_prob", "entities_fraction"]
)
overlap_df = overlap_df.sort_values("overlap_prob", ascending=False)
overlap_df.head(100)

,property,distinct_value_count,overlap_prob,cond_overlap_prob,entities_fraction
27,sport,454,1.559519e-04,0.121602,0.035812
3,country,831,1.047013e-04,0.024069,0.065955
20,instance of,11492,1.147040e-05,0.006363,0.042458
24,located in the administrative territorial entity,53593,6.226884e-06,0.000733,0.092171
6,point in time,13769,4.940307e-06,0.009087,0.023316
...,...,...,...,...,...
165,site of astronomical discovery,129,4.801921e-09,0.075427,0.000252
369,taxon rank,13,4.509965e-09,0.399406,0.000106
321,programmed in,82,4.423756e-09,0.120264,0.000192
284,color,25,4.422222e-09,0.822351,0.000073
